# Sinh bộ câu GPT mới (kế hoạch 06, việc C23 / D16)

Chạy trên **một runtime Colab riêng** (Python và transformers mới của Colab, *không* dùng venv Python 3.8 của `SEARLE_runner.ipynb`). GPU: A100 hoặc L4.

1. Ô 0: chọn chế độ, mô hình, tên bộ câu. **Kiểm tra giấy phép mô hình** trước khi dùng và ghi vào `LICENSE`.
2. Chạy ô 1–4, rồi **ô 5 (chạy thử 20 khái niệm)** xem câu sinh ra có đúng ý không.
3. Ô 6: chạy quy mô nhỏ (khái niệm của shard s0, 64 câu mỗi khái niệm). Tiến độ lưu lên Drive mỗi 500 khái niệm; Colab ngắt thì chạy lại ô 0–4 và ô 6, script tự chạy tiếp.
4. Ô 7: làm sạch (giới hạn 77 token CLIP) và thống kê E6.1. Báo Hòa tên bộ câu để chạy H18.

### Ô 0. Cấu hình

In [ ]:
MODE = "instruct"                              # "continue" = P2 (mô hình nền), "instruct" = P3 (câu thị giác)
MODEL = "microsoft/Phi-3.5-mini-instruct"      # P2 gợi ý: "Qwen/Qwen2.5-1.5B"
LICENSE = "MIT"                                # ghi đúng giấy phép của MODEL (Qwen2.5-1.5B: Apache-2.0)
NAME = "P3_phi35_s0"                           # tên thư mục bộ câu trong data/GPT_phrases/
N_PHRASES = 64
SHARD_NAMES = "oti_B32_s0"                     # quy mô nhỏ: khái niệm của các ảnh trong shard này; "" = toàn bộ khái niệm

### Ô 1. Gắn Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/SEARLE"
assert os.path.isdir(DRIVE), "Chưa thấy MyDrive/SEARLE: thêm lối tắt thư mục SEARLE vào Drive của tôi"

### Ô 2. Lấy code mới nhất

In [ ]:
REPO = "/content/XLA_SEARLE"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin main && git -C {REPO} reset -q --hard origin/main
else:
    !git clone -q https://github.com/chilk1307/XLA_SEARLE.git {REPO}
!git -C {REPO} log -1 --format="Code: %h  %s  (%cr)"

### Ô 3. Thư viện (Python của Colab)

In [ ]:
!pip install -q -U transformers accelerate ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6

### Ô 4. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Ô 5. Chạy thử 20 khái niệm (xem câu trước khi chạy dài)

In [ ]:
SRC = f"{REPO}/SEARLE-main"
SIMILAR = f"{DRIVE}/outputs/similar_concepts/imagenet/test/ViT-B32_top_250_ensembling_image_texts.csv"
SMALL = f"--similar-concepts {SIMILAR} --image-names {DRIVE}/outputs/oti_pseudo_tokens/imagenet/test/{SHARD_NAMES}/image_names.pkl" if SHARD_NAMES else ""
!cd {SRC} && python tools/generate_phrases_llm.py --mode {MODE} --model {MODEL} --license "{LICENSE}" --n-phrases {N_PHRASES} \
    {SMALL} --limit 20 --out-dir /content/test_{NAME}

### Ô 6. Chạy quy mô nhỏ (lưu lên Drive, chạy tiếp được)

In [ ]:
OUT = f"{DRIVE}/outputs/GPT_phrases/{NAME}_raw"
!cd {SRC} && python tools/generate_phrases_llm.py --mode {MODE} --model {MODEL} --license "{LICENSE}" --n-phrases {N_PHRASES} \
    {SMALL} --out-dir {OUT} 2>&1 | tee -a {OUT}.log

### Ô 7. Làm sạch (≤ 77 token CLIP) và thống kê E6.1

In [ ]:
!cd {SRC} && python tools/clean_phrases.py --phrases {OUT}/concept_to_phrases.pkl --out-exp {NAME} --out-root {DRIVE}/outputs/GPT_phrases
!cp {OUT}/hyperparameters.json {DRIVE}/outputs/GPT_phrases/{NAME}/generation.json
!cd {SRC} && python tools/phrases_stats.py --phrases {DRIVE}/outputs/GPT_phrases/{NAME}/concept_to_phrases.pkl \
    --similar-concepts {SIMILAR} --out {DRIVE}/results/E6.1-stats-{NAME}
print(f"Bộ câu sẵn sàng: --gpt-exp-name {NAME}")